[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rudrite/kernels/blob/main/labs/xla/lab-x1-read-your-dump.ipynb)

# LAB·X1 · Read your own dump

**Hardware:** any machine. Everything below runs on CPU.

XLA will hand you a complete, numbered record of every pass it ran, if you ask for it. This lab asks for that record on a real attention program, then reads it the way the compiler wrote it: one file per pass, in order. You will find the exact step where fusion happens, diff the module across that step, and find the one boundary after which every shape in the module is required to carry a concrete physical layout.

Before you run the next cell, guess a number. For one small function, compiled once, how many files land in the dump directory? Ten? Fifty? Write your guess down somewhere. You will check it in a moment.

In [ ]:
import os
os.environ["XLA_FLAGS"] = "--xla_dump_to=/tmp/xla-dump --xla_dump_hlo_pass_re=.*"

import jax
import jax.numpy as jnp

print(jax.__version__, jax.devices())

def attend(q, k, v):
    s = q @ k.T / jnp.sqrt(jnp.float32(q.shape[-1]))
    return jax.nn.softmax(s) @ v

x = jnp.ones((64, 64))
jax.jit(attend).lower(x, x, x).compile()
print("compiled. /tmp/xla-dump now holds one file per pass step.")

**your prediction:**

In [ ]:
DUMP_DIR = "/tmp/xla-dump"
all_files = sorted(os.listdir(DUMP_DIR))
attend_files = sorted(f for f in all_files if "jit_attend" in f)

print(f"{len(all_files)} files total in the dump directory")
print(f"{len(attend_files)} of them belong to attend itself")
print("the rest are helper compiles JAX ran in the same process (constant folding, dtype conversion)")
print()
for f in attend_files[:5]:
    print(" ", f)
print("  ...")
for f in attend_files[-5:]:
    print(" ", f)

On the machine this track was written on, jax 0.4.38 on CPU, this exact driver produced 42 files for `attend`. Your count may differ by a few, depending on your jax version and OS; the shape of the list matters more than the literal number.

Read one filename closely: `module_0005.jit_attend.0017.HLO_passes_after_layout_assignment.after_fusion.before_simplification_after_layout_assignment.txt`. Each numbered file is a full snapshot of the module at that point. The two clauses after the pipeline-bucket name, `after_X` and `before_Y`, say which pass just ran and which one runs next. Read the numbered files in order and you are reading the pipeline itself, not a description of it.

**your prediction:**

In [ ]:
import re
from collections import Counter

pattern = re.compile(r"module_\d+\.jit_attend\.(\d{4})\.(.+)\.txt$")
steps = [pattern.match(f).group(2) for f in attend_files if pattern.match(f)]

counts = Counter(steps)
repeated = {name: n for name, n in counts.items() if n > 1}
print(f"{len(steps)} numbered pass-step files; {len(repeated)} step name(s) appear more than once")
for name, n in repeated.items():
    print(f"  ran {n} times: {name}")

`simplification.after_pipeline-start.before_algsimp` shows up three times. That is not a bug in the dump. Simplification is not one pass, it is a small loop: algebraic simplification, sort simplification, tree-reduction rewriting, and a few others, run in a cycle until nothing changes anymore. Each lap through that loop is its own snapshot. A pass that reruns until it stops finding work is called a fixpoint pass, and this is what a fixpoint pass looks like from the outside: the same boundary, several times, until the loop gives up because there is nothing left to simplify.

**your prediction:**

In [ ]:
fusion_files = [f for f in attend_files if "after_fusion" in f]
print("filename that marks the fusion step:")
for f in fusion_files:
    print(" ", f)

idx = attend_files.index(fusion_files[0])
before_fusion, after_fusion = attend_files[idx - 1], attend_files[idx]
print()
print("before:", before_fusion)
print("after: ", after_fusion)

In [ ]:
import difflib

def read(name):
    with open(os.path.join(DUMP_DIR, name)) as f:
        return f.readlines()

before_lines = read(before_fusion)
after_lines = read(after_fusion)

diff = list(difflib.unified_diff(before_lines, after_lines, lineterm=""))
print(f"{len(diff)} diff lines")
for line in diff[:40]:
    print(line.rstrip())

Before this step, the broadcast and the divide are two separate instructions, each one a full pass over a 64x64 array. After it, they are gone, replaced by a call into a `%fused_computation` with its own parameters and one ROOT instruction. Nothing outside that block ever sees the broadcast's output on its own; the intermediate never gets its own buffer. That is the entire job of fusion: not new math, just fewer trips through memory for the same math.

Here is the same fusion, captured once and pinned for reference (verified, jax 0.4.38, CPU, paths shortened):

```
%fused_computation (param_0: f32[64,64], param_1.1: f32[64]) -> f32[64,64] {
  %param_0 = f32[64,64]{1,0} parameter(0)
  %param_1.1 = f32[64]{0} parameter(1)
  %broadcast.3 = f32[64,64]{1,0} broadcast(f32[64]{0} %param_1.1), dimensions={0}, metadata={op_name="jit(attend)/jit(main)/div"}
  ROOT %divide.0 = f32[64,64]{1,0} divide(f32[64,64]{1,0} %param_0, f32[64,64]{1,0} %broadcast.3), metadata={op_name="jit(attend)/jit(main)/div" source_file="attend.py" source_line=12}
}
```

If your own diff shows the same shape, broadcast feeding straight into an elementwise op, wrapped into one computation, you found the same decision the compiler made on this machine.

**your prediction:**
Which single pass boundary forces every shape in the module to carry a concrete physical layout, and what do you expect the diff across it to show?

In [ ]:
before_layout = next(f for f in attend_files if "before_layout-assignment" in f)
after_layout = next(f for f in attend_files if "after_layout-assignment" in f)
print("before:", before_layout)
print("after: ", after_layout)

layout_diff = list(difflib.unified_diff(read(before_layout), read(after_layout), lineterm=""))
print(f"{len(layout_diff)} diff lines across the layout-assignment boundary")

On this program, that count is zero. The two files are identical. That is a real result, not a broken lab: by the time LayoutAssignment runs, every shape in this module already agrees on a layout, the entry parameters because PJRT fixed their layout before compilation even started, everything else because an earlier pass already settled it. A pass that finds nothing left to decide is still doing its job; it is proving there was no conflict, not skipping its turn.

The layout decision that mattered already happened, earlier in this same dump. Read the very first snapshot, before any pass ran at all, and look for a `transpose` instruction.

In [ ]:
raw_text = "".join(read(next(f for f in attend_files if f.endswith("before_optimizations.txt"))))
settled_text = "".join(read(after_layout))

print("transpose ops in the freshly ingested module:", raw_text.count("transpose("))
print("transpose ops once the module reaches layout-assignment:", settled_text.count("transpose("))

The freshly ingested module carries an explicit `transpose(Arg_1.2), dimensions={1, 0}` with layout `{0,1}`, the literal `k.T` from the Python. By the time the module reaches layout-assignment, that transpose is gone: an earlier pass, transpose-folding, decided the dot's contracting dimension could just point at the other operand's other axis and folded the transpose away entirely. That is a layout decision paying off, in writing, inside your own dump: not a promise about what layouts mean, a disappearance you can point at.

## mark it run

Chapter 04 (kernels.rudrite.com/xla/pipeline) is the dump you just read, argued in prose. Chapter 05 (kernels.rudrite.com/xla/fusion) is the fusion step you found, with three more before-and-after pairs pulled from a TPU. Once your own dump backs up both chapters in writing, LAB·X2 takes the same driver onto a Colab TPU runtime and asks what changes when the hardware does.